In [2]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [4]:
df = pd.read_csv('covid_toy.csv')

In [5]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [6]:
# df['cough'].value_counts()
df['city'].value_counts() # 4 cities

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

Gender and city is the nominal categorical Data : So we need to do the nominal encoding / one hot encoding 
Fever and cough is the ordinal categorical Data : So we need to do Ordinal encoding

Not doing the label encoding on the has_covid, and not doing the scalind too 

In [7]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)


In [9]:
X_train

,age,gender,fever,cough,city
90,59,Female,99.0,Strong,Delhi
98,5,Female,98.0,Strong,Mumbai
41,82,Male,NaN,Mild,Kolkata
67,65,Male,99.0,Mild,Bangalore
79,48,Female,103.0,Mild,Kolkata
...,...,...,...,...,...
30,15,Male,101.0,Mild,Delhi
9,64,Female,101.0,Mild,Delhi
51,11,Female,100.0,Strong,Kolkata
69,73,Female,103.0,Mild,Delhi


### Without the Column transformer . 

In [10]:

# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [11]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']]) # smaller to larger value
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape
# X_train_cough

(80, 1)

In [12]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [13]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

Appending the new transformed columns

In [14]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

# X_train_transformed
X_train_transformed.shape

(80, 7)

### With the Use of the Column Transformer

In [57]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder

In [ ]:
transformer = ColumnTransformer(transformers=[
    ('imputed', SimpleImputer(), ['fever']), # we pass the tuple , with args : name of the transformer, Transformer Object , List of the columns where we want the transformation
    ('o_encoded', OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']),
    ('ohe', OneHotEncoder(sparse_output=False,drop='first'), ['gender','city']),
    # ('le',LabelEncoder(),['has_covid']) this doesnot work
],remainder='passthrough') # remainder is the column that doesnot need the transformation so passthrough and drop option exists we want age to be as it is

In [66]:
Xtrain_new=transformer.fit_transform(X_train)
# transformer.fit_transform(X_train).shape

ValueError: Some column names are not columns of the dataframe: {'has_covid'}

##### So we got the same transformed array with the column transformation so easily 

In [35]:
# transformer.transform(X_test).shape
Xtest_new=transformer.transform(X_test)

This becomes the np array . so converting back to df . 

Getting Feature Names

After one-hot encoding, many new columns are created.

In [67]:
transformer.fit(df)

transformer.get_feature_names_out()

# this name is very useful for converting the transformed array back to data Frame 

TypeError: LabelEncoder.fit_transform() takes 2 positional arguments but 3 were given

In [64]:
X=transformer.fit_transform(df)
df_new = pd.DataFrame(
    X,
    columns=transformer.get_feature_names_out()
)

print(df_new)

TypeError: LabelEncoder.fit_transform() takes 2 positional arguments but 3 were given

In [32]:
df_new.head()

,imputed__fever,o_encoded__cough,ohe__gender_Male,ohe__city_Delhi,ohe__city_Kolkata,ohe__city_Mumbai,remainder__age,remainder__has_covid
0,103.0,0.0,1.0,0.0,1.0,0.0,60,No
1,100.0,0.0,1.0,1.0,0.0,0.0,27,Yes
2,101.0,0.0,1.0,1.0,0.0,0.0,42,No
3,98.0,0.0,0.0,0.0,1.0,0.0,31,No
4,101.0,0.0,0.0,0.0,0.0,1.0,65,No


Now training is so easy as preprocessing is done so easily

In [54]:
type(df_new)
df_new.columns

Index(['imputed__fever', 'o_encoded__cough', 'ohe__gender_Male',
       'ohe__city_Delhi', 'ohe__city_Kolkata', 'ohe__city_Mumbai',
       'remainder__age', 'remainder__has_covid'],
      dtype='object')

In [55]:
X_trainnew,X_testnew,y_trainnew,y_testnew = train_test_split(df_new.drop(columns=['remainder__has_covid']),df_new['remainder__has_covid'],
                                                test_size=0.2)

In [56]:
y_testnew

51    Yes
1     Yes
81     No
70     No
77     No
78    Yes
88     No
0      No
55    Yes
2      No
76    Yes
13    Yes
61     No
24     No
36     No
22    Yes
54    Yes
49     No
29    Yes
80    Yes
Name: remainder__has_covid, dtype: object

In [33]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()

In [36]:
model.fit(Xtrain_new, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [38]:
pred = model.predict(Xtest_new)
pred

array(['No', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes',
       'No', 'Yes', 'No', 'No', 'No', 'No', 'Yes', 'No', 'Yes', 'No'],
      dtype=object)

In [39]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test,pred)

0.4